# 03 — The Greeks

The Greeks are partial derivatives of the option price with respect to each input. They quantify the sensitivity of a position and are the core tools for risk management.

| Greek | Derivative | Intuition |
|-------|-----------|----------|
| Delta $\Delta$ | $\partial C / \partial S$ | Price change per $1 move in stock |
| Gamma $\Gamma$ | $\partial^2 C / \partial S^2$ | Rate of change of Delta |
| Theta $\Theta$ | $\partial C / \partial t$ | Daily time decay |
| Vega $\mathcal{V}$ | $\partial C / \partial \sigma$ | Price change per 1% move in vol |
| Rho $\rho$ | $\partial C / \partial r$ | Price change per 1% move in rate |

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
from greeks import delta, gamma, theta, vega, rho

In [ ]:
S     = 150.0
K     = 150.0
T     = 0.25
r     = 0.05
sigma = 0.20

print("Greeks at baseline parameters (ATM call):")
print(f"  Delta : {delta(S, K, T, r, sigma):.4f}")
print(f"  Gamma : {gamma(S, K, T, r, sigma):.4f}")
print(f"  Theta : {theta(S, K, T, r, sigma):.4f}  (per calendar day)")
print(f"  Vega  : {vega(S, K, T, r, sigma):.4f}  (per 1% vol move)")
print(f"  Rho   : {rho(S, K, T, r, sigma):.4f}   (per 1% rate move)")

## All Greeks vs. stock price $S$
The most important plot: how each sensitivity evolves as the option moves from out-of-the-money to in-the-money.

In [ ]:
S_range = np.linspace(80, 220, 300)

greeks_data = {
    'Delta':  [delta(s, K, T, r, sigma) for s in S_range],
    'Gamma':  [gamma(s, K, T, r, sigma) for s in S_range],
    'Theta':  [theta(s, K, T, r, sigma) for s in S_range],
    'Vega':   [vega(s, K, T, r, sigma)  for s in S_range],
    'Rho':    [rho(s, K, T, r, sigma)   for s in S_range],
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
colors = ['steelblue', 'darkorange', 'tomato', 'seagreen', 'mediumpurple']

for ax, (name, values), color in zip(axes, greeks_data.items(), colors):
    ax.plot(S_range, values, color=color)
    ax.axvline(K, color='black', linestyle=':', linewidth=1)
    ax.set_title(name)
    ax.set_xlabel('Stock price S')

axes[-1].axis('off')  # hide empty 6th subplot
fig.suptitle('Greeks vs. underlying price (ATM strike = 150)', fontsize=13)
plt.tight_layout()
plt.show()

## Delta heatmap: $S$ vs. $T$
Delta changes over time as well as with the stock price. Near expiry, ATM delta stays near 0.5 but ITM/OTM deltas collapse to 1 or 0.

In [ ]:
S_grid = np.linspace(100, 200, 80)
T_grid = np.linspace(0.02, 1.0, 80)   # 1 week to 1 year
SS, TT = np.meshgrid(S_grid, T_grid)

delta_map = np.vectorize(lambda s, t: delta(s, K, t, r, sigma))(SS, TT)

fig, ax = plt.subplots(figsize=(9, 6))
c = ax.contourf(S_grid, T_grid, delta_map, levels=50, cmap='RdYlGn')
fig.colorbar(c, ax=ax, label='Delta')
ax.axvline(K, color='black', linestyle='--', linewidth=1, label=f'Strike K={K}')
ax.set_xlabel('Stock price S')
ax.set_ylabel('Time to expiry T (years)')
ax.set_title('Call Delta heatmap: S vs. T')
ax.legend()
plt.tight_layout()
plt.show()

## Theta decay over time
Time decay is not linear — it accelerates as expiry approaches. An ATM option loses value faster in its final weeks.

In [ ]:
T_range  = np.linspace(0.01, 1.0, 300)
theta_atm = [theta(S, K, t, r, sigma) for t in T_range]           # at-the-money
theta_otm = [theta(S, K * 1.15, t, r, sigma) for t in T_range]    # 15% out-of-the-money

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(T_range, theta_atm, label='ATM (K=150)',  color='steelblue')
ax.plot(T_range, theta_otm, label='OTM (K=172.5)', color='tomato')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Time to expiry T (years)')
ax.set_ylabel('Theta ($/day)')
ax.set_title('Theta decay vs. time to expiry')
ax.legend()
plt.tight_layout()
plt.show()